# MSIN0097: Agentic AI Comparative Analysis

This notebook tracks the autonomous performance of three AI coding agents — **Claude (Anthropic)**, **Codex (OpenAI)**, and **Antigravity** — as they independently ingest, clean, and explore the US DOT 2015 Aviation Dataset (~5.8 million domestic flight records). Each agent received identical task briefs and was evaluated on domain-logic correctness, code quality, memory efficiency, and resistance to ML anti-patterns such as look-ahead data leakage. The cells below present a structured comparison of their outputs, failure modes, and self-correction behaviours across Task 1 (Data Ingestion & Missingness) and Task 1.5 (Exploratory Data Analysis).

## Task 1: Dataset Ingestion & Missingness (The Domain Logic Trap)

All three agents were asked to load the raw flight data, perform LEFT JOIN enrichment using `airlines.csv` and `airports.csv`, handle missing values in the `ARRIVAL_DELAY` target, and apply memory-efficient dtypes. The critical domain-logic test was whether the agent understood that `ARRIVAL_DELAY = NaN` for a **cancelled flight** is not a statistical missingness problem — it is a **structurally undefined value** that should be excluded from a delay-regression dataset entirely.

| Agent | JOIN Logic | Missingness Strategy | Memory Optimisation | Sampling (500k) |
|---|---|---|---|---|
| **Claude** (Apple Silicon) | ✅ Correct LEFT JOIN | ✅ Dropped cancelled flights (domain pass) | ✅ Autonomous downcast to `float32`/`int8` — **83% compression** | ❌ Agentic Override: loaded full 5.8M rows |
| **Codex** (macOS) | ✅ Cleanest join logic | ✅ Correctly identified structural NaN for cancelled flights | ⚠️ Standard dtypes, no aggressive downcast | ✅ Only agent to attempt 500k sampling |
| **Antigravity** (Windows) | ✅ Correct LEFT JOIN | ❌ **Fatal domain logic failure**: applied `SimpleImputer(strategy='median')` to cancelled flights, mathematically forcing crashed/cancelled planes to arrive | ⚠️ Basic optimisation | ❌ No sampling attempted |

**Key finding:** Antigravity's use of `SimpleImputer(strategy='median')` on cancelled flights represents a fundamental domain-logic error. Imputing an arrival delay for a flight that never departed is statistically nonsensical and would silently corrupt any downstream model trained on this data. Claude correctly applied the domain rule (drop, don't impute) while also autonomously discovering and applying aggressive dtype compression beyond what was explicitly requested — an example of beneficial agentic initiative, offset by its failure to respect the row-count constraint.

## Task 1.5: Exploratory Data Analysis (The Leakage Trap)

Agents were asked to produce three sophisticated EDA plots investigating the drivers of delays `> 15 minutes`, with the constraint that **only features known 24 hours in advance** (pre-departure) may be used — explicitly excluding `DEPARTURE_DELAY`, `TAXI_OUT`, and real-time operational columns.

| Agent | Leakage Avoidance | Plot Sophistication | Business Insight Comment |
|---|---|---|---|
| **Claude** | ✅ No leakage | ✅✅ **Distinction-level** — used 95% Wilson confidence intervals for binary proportions; produced carrier×month heatmap, temporal danger-zone heatmap, and distance×time-of-day interaction plot | ✅ Complete multi-line insight block |
| **Codex** | ✅ No leakage | ✅✅ Highly advanced — multivariate scatter plot mapping route volume vs. delay rate with bubble sizing | ❌ **Context-dropout failure**: did not write the requested business insight comment at script end |
| **Antigravity** | ✅ No leakage | ⚠️ Structurally basic plots | ✅ Present and accurate |

**Notable behaviour:** All three agents independently passed the look-ahead leakage trap — none used `DEPARTURE_DELAY` as a predictor during EDA, despite it being the strongest single correlate of `ARRIVAL_DELAY` in the dataset. Antigravity autonomously self-corrected its Task 1 imputation error by dropping cancelled flights *before* plotting, suggesting the agent detected the inconsistency without being explicitly prompted. Codex's context-dropout (failing to append the insight comment) is a known failure mode for instruction-following on long code-generation tasks.

### EDA Visual Evidence

---

#### Claude — Plot 1: Carrier × Month Delay-Rate Heatmap
![Claude EDA Plot 1: Carrier × Month Heatmap](agent_outputs/claude_code/eda_plot_01_carrier_month_heatmap.png)

---

#### Claude — Plot 2: Temporal Danger Zones (Day-of-Week × Departure Hour)
![Claude EDA Plot 2: Temporal Danger Zones](agent_outputs/claude_code/eda_plot_02_temporal_danger_zones.png)

---

#### Claude — Plot 3: Distance Quintile × Time-of-Day Delay Rate
![Claude EDA Plot 3: Distance × Time-of-Day](agent_outputs/claude_code/eda_plot_03_distance_tod_delay.png)

---

#### Codex — Plot 1: Delay Risk by Scheduled Departure Hour and Day of Week
![Codex EDA Plot 1: Delay Heatmap Hour × Day](agent_outputs/codex/plot_1_delay_heatmap_hour_day.png)

---

#### Codex — Plot 2: High-Risk Routes — Delay Rate vs Volume
![Codex EDA Plot 2: Route Risk Frontier](agent_outputs/codex/plot_2_route_risk_frontier.png)

---

#### Codex — Plot 3: Seasonal Delay Exposure for the Largest Airlines
![Codex EDA Plot 3: Airline Seasonality](agent_outputs/codex/plot_3_airline_seasonality.png)

---

#### Antigravity — Plot 1: Airline Delay Rates
![Antigravity EDA Plot 1: Airline Delays](agent_outputs/antigravity/eda_plot1_airline_delays.png)

---

#### Antigravity — Plot 2: Delays by Departure Hour
![Antigravity EDA Plot 2: Hour Delays](agent_outputs/antigravity/eda_plot2_hour_delays.png)

---

#### Antigravity — Plot 3: Delay Rate Heatmap
![Antigravity EDA Plot 3: Heatmap](agent_outputs/antigravity/eda_plot3_heatmap.png)

## Task 2: Comparative EDA Evaluation

Agents were given four structured EDA experiments on the full 5.8M-row dataset (subsampled to 500k rows): **(1)** Worst Offenders by delay rate, **(2)** Descriptive statistics for the skewed target, **(3)** Correlation analysis and Day-of-Week × Airline pivot table, and **(4)** Advanced statistics — VIF multicollinearity audit and heteroscedasticity diagnosis. Two traps were embedded: the **Volume Trap** (ranking by raw count rather than rate) in Experiment 1, and the **Look-Ahead Leakage Trap** (reintroducing `DEPARTURE_DELAY` into the VIF feature set) in Experiment 4.

---

### Task 2 Comparative Matrix

| Dimension | Claude | Codex | Antigravity |
|---|---|---|---|
| **Worst Offenders — rate not volume** | ✅ Pure delay rate; min-count guard; airline + airport dual panel | ✅✅ **Empirical Bayes posterior** (Beta conjugate prior) with 95% credible intervals — highest statistical rigour of all agents | ❌ `severity = rate × mean_delay` — invalid composite metric; no minimum flight count filter |
| **Descriptive stats — P90 + P99 present** | ✅ P90, P99 + skewness, kurtosis | ⚠️ P95 only — **P99 missing** from requested percentile set | ✅ P90, P95, P99, skewness, kurtosis — most complete table |
| **Correlation — categorical/ID exclusion** | ✅ Manual exclusion list: HHMM ints, breakdown columns, ID cols | ✅ Spearman (rank-robust) for numeric + **Correlation Ratio η for categorical** (AIRLINE, ORIGIN_AIRPORT); excludes delay breakdown block — methodologically superior | ❌ Raw `select_dtypes(include=number)` — no exclusions; admits DEPARTURE_DELAY, FLIGHT_NUMBER, YEAR (constant), all HHMM circular ints |
| **VIF — no crash, no leakage in feature set** | ✅ Pre-departure features only; near-zero variance guard; `sm.add_constant` | ✅ Custom `clean_numeric_matrix`: drops inf, null-cols, constant cols, median-imputes; pre-departure features only | ❌ **`DEPARTURE_DELAY` in VIF features** — post-departure leakage reintroduced; corrupts multicollinearity finding |
| **Heteroscedasticity — formal test + plot** | ✅ Breusch-Pagan (p=8.47e-35); residuals vs fitted + scale-location; LOWESS overlay | ✅ Hexbin density + binned mean-abs-residual overlay; quantified `corr` and `slope` as numeric outputs; sklearn Pipeline approach | ✅ Breusch-Pagan computed; residuals vs fitted plot; results written to `.txt` — but OLS fitted on leaky feature set |
| **Business insight — actionable, named** | ✅ Skewness/kurtosis quantified; DISTANCE ↔ SCHEDULED_TIME collinearity flagged by name; heteroscedasticity linked to log-transform recommendation | ✅ Names `worst_airline`, `second_worst`, `weekday_hotspot` from live data variables; bridges heteroscedasticity → Task 3 model selection (boosted trees) | ❌ Generic tautologies: "avoid bad carriers", "reduce exposure to extreme delays" — no named entity, no quantified threshold |
| **Code documentation — explains "why"** | ✅ Thread-limiting env vars with rationale; LOWESS smoother purpose stated | ⚠️ One genuine "why" comment (VIF singularity guard); Empirical Bayes block uncommented despite being the most novel section | ❌ All "what" comments; no statistical rationale for metric choice, test selection, or leakage scope |
| **Leakage discipline — consistent across all scripts** | ✅ No leakage in any script | ✅ No leakage in any script | ❌ **Leakage-free in basic EDA; violated in advanced stats** — `DEPARTURE_DELAY` reintroduced inconsistently |
| **Overall Grade** | **Merit** | **Distinction** | **Pass (borderline)** |

---

### Verdict

**Codex wins the Task 2 EDA phase.**

The decisive differentiator was the **Empirical Bayes posterior** applied to the Worst Offenders analysis. By placing a Beta conjugate prior — calibrated from the fleet-wide severe-delay rate with strength proportional to median airline sample size — Codex produced the only ranking that is statistically defensible for a travel-agency client: small carriers with sparse data are correctly shrunk toward the baseline rather than artificially inflated. No other agent attempted this level of statistical rigour. Codex also uniquely handled categorical features in its correlation analysis using the **Correlation Ratio (η)**, the correct association measure for a mixed numeric/categorical feature set against a continuous target. The advanced stats script bridges EDA findings quantitatively to Task 3 model selection, naming boosted trees and explaining *why* they are appropriate given the residual structure — not as a default recommendation but as a consequence of the heteroscedasticity and non-linearity demonstrated in this script.

Claude produced clean, production-grade work across all four experiments and correctly implemented the Breusch-Pagan test with a LOWESS diagnostic, but its worst-offenders plot and correlation table used simpler methodologies without uncertainty quantification. Antigravity showed a fatal inconsistency: it avoided the Look-Ahead Leakage trap in basic EDA (as in Task 1.5) but then reintroduced `DEPARTURE_DELAY` into the VIF feature set in `task2_advanced_stats.py`. This reveals that Antigravity does not hold the 24-hour prediction constraint as a persistent domain rule — it only applies it when the task brief explicitly mentions leakage, not as an internalized modelling principle. The composite severity metric and fully-contaminated correlation matrix further disqualify its advanced EDA from the top grade.

### Task 2 Visual Evidence

<table>
<thead>
<tr>
  <th style="text-align:center; width:33%">Claude</th>
  <th style="text-align:center; width:33%">Codex</th>
  <th style="text-align:center; width:33%">Antigravity</th>
</tr>
</thead>
<tbody>
<tr>
  <td colspan="3" style="text-align:center; background:#f0f0f0; font-weight:bold; padding:6px">
    Worst Offenders (Experiment 1) — Delay Rate Ranking
  </td>
</tr>
<tr>
  <td><img src="agent_outputs/claude/eda_worst_offenders.png" width="100%" alt="Claude Worst Offenders"/></td>
  <td><img src="agent_outputs/codex/eda_worst_offenders.png" width="100%" alt="Codex Worst Offenders"/></td>
  <td><img src="agent_outputs/antigravity/eda_worst_offenders.png" width="100%" alt="Antigravity Worst Offenders"/></td>
</tr>
<tr>
  <td style="text-align:center; font-size:0.85em">Pure delay rate, dual panel (airline + airport)</td>
  <td style="text-align:center; font-size:0.85em">Empirical Bayes posterior with 95% credible intervals</td>
  <td style="text-align:center; font-size:0.85em">Composite severity = rate × mean (methodologically invalid)</td>
</tr>
<tr>
  <td colspan="3" style="text-align:center; background:#f0f0f0; font-weight:bold; padding:6px">
    Heteroscedasticity Diagnostic (Experiment 4) — Residuals vs Fitted
  </td>
</tr>
<tr>
  <td><img src="agent_outputs/claude/eda_heteroscedasticity.png" width="100%" alt="Claude Heteroscedasticity"/></td>
  <td><img src="agent_outputs/codex/eda_heteroscedasticity.png" width="100%" alt="Codex Heteroscedasticity"/></td>
  <td><img src="agent_outputs/antigravity/eda_heteroscedasticity.png" width="100%" alt="Antigravity Heteroscedasticity"/></td>
</tr>
<tr>
  <td style="text-align:center; font-size:0.85em">Breusch-Pagan p=8.47e-35; residuals + scale-location; LOWESS overlay</td>
  <td style="text-align:center; font-size:0.85em">Hexbin density + binned mean absolute residual; sklearn Pipeline</td>
  <td style="text-align:center; font-size:0.85em">Breusch-Pagan computed but OLS fitted on leaky features (incl. DEPARTURE_DELAY)</td>
</tr>
</tbody>
</table>